# VLM-Anomaly — Full MVTec Sweep · PatchCore (Classical Baseline)

**Model:** `patchcore` via [Anomalib](https://github.com/openvinotoolkit/anomalib) (Intel, Apache 2.0)  
**Strategy:** Train on normal split → evaluate on test split.  
**Cost:** **$0** — fully open-source.  
**GPU:** Auto-detects Apple MPS (AMD Radeon / Apple Silicon) via `torch.backends.mps.is_available()`.

| Scope | Est. time (CPU) | Est. time (MPS/GPU) | Cost |
|---|---|---|---|
| 1 category | ~3–5 min | ~30–60 s | $0 |
| Full sweep (15 cat) | ~45–75 min | ~8–15 min | $0 |

In [1]:
# ── Cell 1: Setup paths & sys.path ─────────────────────────────────────────
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT   = Path().resolve().parent
SRC_DIR     = REPO_ROOT / 'src'
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert SRC_DIR.exists(), f'src/ not found at {SRC_DIR}'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(REPO_ROOT / '.env')

import vlm_anomaly
print(f'vlm_anomaly {vlm_anomaly.__version__} ready')
print(f'Results : {RESULTS_DIR}')


vlm_anomaly 0.1.0 ready
Results : /Users/sabareeswarans/Projects_26/VLM-Anomaly/results


In [2]:
# ── Cell 2: Verify anomalib + torch ─────────────────────────────────────────
try:
    import anomalib, torch
    print(f'anomalib {anomalib.__version__} | torch {torch.__version__}')
except ImportError as e:
    raise ImportError(
        f'Missing dependency: {e}\n'
        'Install with: pip install anomalib torch==2.2.2 torchvision timm\n'
        'Or: uv pip install -e \".[classical]\"'
    )


anomalib 2.4.2 | torch 2.2.2


In [3]:
# ── Cell 3: Find MVTec dataset ──────────────────────────────────────────────
MVTEC_ROOT = None
for candidate in [
    REPO_ROOT / 'data' / 'mvtec',
    REPO_ROOT / 'data' / 'mvtec-ad',
    Path('/tmp/mvtec'),
]:
    if candidate.exists() and any(candidate.iterdir()):
        MVTEC_ROOT = candidate
        break

assert MVTEC_ROOT, f'MVTec not found. Expected at {REPO_ROOT}/data/mvtec'
categories = sorted([d.name for d in MVTEC_ROOT.iterdir() if d.is_dir()])
print(f'MVTec root : {MVTEC_ROOT}')
print(f'Categories : {categories}')


MVTec root : /Users/sabareeswarans/Projects_26/VLM-Anomaly/data/mvtec
Categories : ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


In [ ]:
# ── Cell 4: Configure ───────────────────────────────────────────────────────
import torch

MODEL_NAME  = 'patchcore'
MODEL_ID    = f'classical/{MODEL_NAME}'
IMAGE_SIZE  = 256
DATASET     = 'mvtec'

# Auto-detect MPS (Apple Metal — works on AMD Radeon AND Apple Silicon).
# Falls back to CPU if MPS is unavailable.
if torch.backends.mps.is_available():
    ACCELERATOR = 'mps'
elif torch.cuda.is_available():
    ACCELERATOR = 'gpu'
else:
    ACCELERATOR = 'cpu'

print(f'Model       : {MODEL_NAME}  ({MODEL_ID})')
print(f'Img size    : {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Accelerator : {ACCELERATOR}')
print(f'Cost        : $0 (open-source)')

In [5]:
# ── Cell 5: Build shared objects ────────────────────────────────────────────
from vlm_anomaly.config import Settings
from vlm_anomaly.logging import configure_logging
from vlm_anomaly.evaluators.classical_evaluator import ClassicalEvaluator

configure_logging(log_level='INFO')

settings = Settings(
    _env_file=str(REPO_ROOT / '.env'),
    data_dir=str(MVTEC_ROOT.parent),
    results_dir=str(RESULTS_DIR),
)

print(f'Settings ready — data_dir={settings.data_dir}')


Settings ready — data_dir=/Users/sabareeswarans/Projects_26/VLM-Anomaly/data


In [ ]:
# ── Cell 5b: Smoke test — 1 category before full sweep ──────────────────────
# Optional — run this to verify the setup before the full 15-category sweep.
import time, json as _json

smoke_cat = 'bottle'
print(f'Smoke test: {MODEL_NAME} / {smoke_cat} / {ACCELERATOR} ...')
t0 = time.perf_counter()
ev = ClassicalEvaluator(
    model_name=MODEL_NAME, dataset_name=DATASET,
    category=smoke_cat, image_size=IMAGE_SIZE,
    accelerator=ACCELERATOR, settings=settings,
)
smoke_result = ev.run()
print(f'  AUROC    : {smoke_result.auroc:.4f}')
print(f'  F1       : {smoke_result.f1:.4f}')
print(f'  elapsed  : {(time.perf_counter()-t0):.0f}s')
print(f'  cost     : $0')
print('PASS' if smoke_result.auroc and smoke_result.auroc > 0.5 else 'FAIL — check setup')

In [ ]:
# ── Cell 6: Run all 15 categories (idempotent, resumable) ───────────────────
import json as _json
from tqdm.auto import tqdm

all_results = []


def _already_done(results_dir, category, model_id, dataset):
    """Return path if a complete result file for this model+category exists."""
    for f in results_dir.glob(f'*_{dataset}_{category}_*.json'):
        try:
            data = _json.loads(f.read_text())
            rows = data if isinstance(data, list) else [data]
            if any(r.get('model_id') == model_id for r in rows):
                return f
        except Exception:
            pass
    return None


for category in tqdm(categories, desc=f'{MODEL_NAME} sweep ({ACCELERATOR})'):
    done = _already_done(RESULTS_DIR, category, MODEL_ID, DATASET)
    if done:
        print(f'  [skip] {category} — already done ({done.name})')
        data = _json.loads(done.read_text())
        all_results.extend(data if isinstance(data, list) else [data])
        continue

    ev = ClassicalEvaluator(
        model_name=MODEL_NAME, dataset_name=DATASET,
        category=category, image_size=IMAGE_SIZE,
        accelerator=ACCELERATOR, settings=settings,
    )
    result = ev.run()
    all_results.append(result.model_dump())
    print(
        f'  {category:12s}  AUROC={result.auroc:.3f}  F1={result.f1:.3f}  '
        f'elapsed={result.mean_latency_ms/1000:.0f}s  [{ACCELERATOR}]'
    )

print(f'\nDone — {len(all_results)} categories processed.')

In [ ]:
# ── Cell 7: Summary table ────────────────────────────────────────────────────
import pandas as pd

if all_results:
    df = pd.DataFrame(all_results)
    num_cols = ['auroc','f1','precision','recall','mean_latency_ms']
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    print(f'Model    : {MODEL_ID}')
    print(f'Dataset  : {DATASET}')
    print(f'Mean AUROC : {df.auroc.mean():.4f}')
    print(f'Mean F1    : {df.f1.mean():.4f}')
    print(f'Avg time/cat : {df.mean_latency_ms.mean()/1000:.0f}s')
    print(f'Total cost : $0.00  (open-source)')
    print()
    display(df[['category','auroc','f1','mean_latency_ms']].sort_values('auroc', ascending=False).reset_index(drop=True))
else:
    print('No results yet — run Cell 6 first.')


In [ ]:
# ── Cell 8: Generate / update report ─────────────────────────────────────────
import importlib, vlm_anomaly.analysis.report_generator as _rg_mod
importlib.reload(_rg_mod)
from vlm_anomaly.analysis.report_generator import generate

REPORT = REPO_ROOT / 'REPORT.md'
generate(RESULTS_DIR, REPORT)
print(f'Report written → {REPORT}')


In [ ]:
# ── Cell 9: Show result files + commit hint ──────────────────────────────────
result_files = sorted(RESULTS_DIR.glob(f'*_{DATASET}_*_{MODEL_NAME}.json'))
print(f'Result files ({len(result_files)}):')
for f in result_files:
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')

# print()
# print('To commit:')
# print(f'  git add results/*_{MODEL_NAME}.json')
# print(f'  git commit -m "results({MODEL_ID}): MVTec sweep"')
